###Initializing Catalog, Schema and Volume

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS quickcart;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS quickcart.default;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS quickcart.default.source_data;

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

from datetime import datetime, timedelta
import random
import uuid

###Customer Data

In [0]:


PROJECT_NAME = "quickcart"

# Unity Catalog objects
CATALOG = "quickcart"

# Source data location
SOURCE_VOLUME = f"/Volumes/{CATALOG}/default/source_data"

# Individual source locations
CUSTOMER_PATH = f"{SOURCE_VOLUME}/customers"
PRODUCT_PATH = f"{SOURCE_VOLUME}/products"
ORDER_PATH = f"{SOURCE_VOLUME}/orders"
PAYMENT_PATH = f"{SOURCE_VOLUME}/payments"
DELIVERY_PATH = f"{SOURCE_VOLUME}/deliveries"

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

print(f"Project       : {PROJECT_NAME}")
print(f"Source Volume : {SOURCE_VOLUME}")

Project       : quickcart
Source Volume : /Volumes/quickcart/default/source_data


In [0]:
num_customers = 100_000
customer_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("date_of_birth", DateType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("pincode", StringType(), True),
    StructField("registration_date", DateType(), True),
    StructField("customer_segment", StringType(), True),
    StructField("updated_at", TimestampType(), True)
])


# ------------------------------------------------------------
# Reference data
# ------------------------------------------------------------

first_names = [
    "Rahul", "Amit", "Rohan", "Vikas", "Akash",
    "Suresh", "Raj", "Arjun", "Karan", "Vivek",
    "Priya", "Neha", "Sneha", "Pooja", "Anjali",
    "Kavya", "Aarti", "Simran", "Riya", "Nisha"
]

last_names = [
    "Sharma", "Patil", "Kumar", "Singh", "Mehta",
    "Desai", "Joshi", "Gupta", "Verma", "Shah",
    "Pawar", "Kulkarni", "Jadhav", "Mishra", "Reddy"
]

locations = [
    ("Mumbai", "Maharashtra", "400001"),
    ("Pune", "Maharashtra", "411001"),
    ("Nagpur", "Maharashtra", "440001"),
    ("Nashik", "Maharashtra", "422001"),
    ("Delhi", "Delhi", "110001"),
    ("Bangalore", "Karnataka", "560001"),
    ("Hyderabad", "Telangana", "500001"),
    ("Chennai", "Tamil Nadu", "600001"),
    ("Kolkata", "West Bengal", "700001"),
    ("Ahmedabad", "Gujarat", "380001"),
    ("Jaipur", "Rajasthan", "302001"),
    ("Surat", "Gujarat", "395001")
]

segments = [
    "Regular",
    "Premium",
    "VIP"
]

genders = [
    "Male",
    "Female"
]

# ------------------------------------------------------------
# Generate customer records
# ------------------------------------------------------------
customer_data = []

start_date = datetime(2022, 1, 1)
end_date = datetime(2026, 7, 31)

for i in range(1, num_customers+1):
    first_name = random.choice(first_names)
    last_name = random.choice(last_names)

    customer_name = f"{first_name} {last_name}"

    customer_id = f"C{i:06d}"

    email = (
        f"{first_name.lower()}."
        f"{last_name.lower()}"
        f"{i}@quickcart.com"
    )

    phone = f"{random.randint(6000000000, 9999999999)}"

    gender = random.choice(genders)

    # Age between approximately 18 and 65
    dob = datetime(
        random.randint(1960, 2006),
        random.randint(1, 12),
        random.randint(1, 28)
    ).date()

    city, state, pincode = random.choice(locations)

    registration_date = (
        start_date +
        timedelta(
            days=random.randint(
                0,
                (end_date - start_date).days
            )
        )
    ).date()

    segment = random.choices(
        segments,
        weights=[70, 25, 5],
        k=1
    )[0]

    updated_at = datetime(
        2026,
        random.randint(1, 7),
        random.randint(1, 28),
        random.randint(0, 23),
        random.randint(0, 59),
        random.randint(0, 59)
    )

    customer_data.append((
        customer_id,
        customer_name,
        email,
        phone,
        gender,
        dob,
        city,
        state,
        pincode,
        registration_date,
        segment,
        updated_at
    ))

print(f"Generated {len(customer_data):,} customer records")

Generated 100,000 customer records


In [0]:
df_customer = spark.createDataFrame(customer_data, customer_schema)
display(df_customer.limit(5))

#df_customer.groupBy("customer_id").count().filter(f.col("count")>1).show()
'''
df_customer.select([
    F.sum(
        F.when(f.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df_customer.columns
]).show()
'''
df_customer.repartition(10).write.mode("overwrite").parquet(CUSTOMER_PATH)

customer_id,customer_name,email,phone,gender,date_of_birth,city,state,pincode,registration_date,customer_segment,updated_at
C000001,Akash Kulkarni,akash.kulkarni1@quickcart.com,6440213415,Male,1997-07-02,Mumbai,Maharashtra,400001,2022-07-11,Regular,2026-05-20T00:35:12.000Z
C000002,Simran Joshi,simran.joshi2@quickcart.com,6946785248,Female,1997-05-26,Mumbai,Maharashtra,400001,2026-04-04,Premium,2026-06-14T10:17:09.000Z
C000003,Raj Jadhav,raj.jadhav3@quickcart.com,7445662585,Male,1965-07-04,Bangalore,Karnataka,560001,2023-12-06,Regular,2026-07-02T23:29:34.000Z
C000004,Vikas Reddy,vikas.reddy4@quickcart.com,7625792787,Male,1995-05-27,Jaipur,Rajasthan,302001,2025-06-20,Premium,2026-03-19T06:45:04.000Z
C000005,Amit Pawar,amit.pawar5@quickcart.com,6978815630,Female,1965-04-28,Pune,Maharashtra,411001,2024-02-18,Regular,2026-06-27T11:10:23.000Z


In [0]:
#display(spark.read.parquet(CUSTOMER_PATH).limit(5))

customer_id,customer_name,email,phone,gender,date_of_birth,city,state,pincode,registration_date,customer_segment,updated_at
C000005,Amit Pawar,amit.pawar5@quickcart.com,6978815630,Female,1965-04-28,Pune,Maharashtra,411001,2024-02-18,Regular,2026-06-27T11:10:23.000Z
C000015,Simran Jadhav,simran.jadhav15@quickcart.com,9135030058,Male,2005-05-13,Jaipur,Rajasthan,302001,2025-08-23,Regular,2026-05-15T03:15:14.000Z
C000025,Priya Singh,priya.singh25@quickcart.com,7140805649,Female,1968-11-21,Delhi,Delhi,110001,2024-07-25,Regular,2026-07-03T00:29:39.000Z
C000035,Simran Desai,simran.desai35@quickcart.com,6118543408,Male,1976-03-19,Delhi,Delhi,110001,2022-03-20,Regular,2026-04-12T23:50:20.000Z
C000045,Arjun Mishra,arjun.mishra45@quickcart.com,8738735023,Female,1993-08-21,Nashik,Maharashtra,422001,2023-07-17,Regular,2026-06-10T07:17:21.000Z


###Products Data

In [0]:
# ============================================================
# PRODUCT DATA GENERATION
# ============================================================

import builtins

NUM_PRODUCTS = 5_000

product_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("subcategory", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("cost", DoubleType(), True),
    StructField("supplier_id", StringType(), True),
    StructField("product_rating", DoubleType(), True),
    StructField("created_date", DateType(), True),
    StructField("updated_at", TimestampType(), True)
])

# ------------------------------------------------------------
# Reference data
# ------------------------------------------------------------

product_categories = {
    "Electronics": [
        "Mobile",
        "Laptop",
        "Tablet",
        "Headphones",
        "Smartwatch",
        "Camera"
    ],
    "Fashion": [
        "Men Clothing",
        "Women Clothing",
        "Shoes",
        "Bags",
        "Accessories"
    ],
    "Home & Kitchen": [
        "Furniture",
        "Kitchen Appliances",
        "Home Decor",
        "Cookware",
        "Storage"
    ],
    "Beauty": [
        "Skincare",
        "Haircare",
        "Makeup",
        "Fragrance"
    ],
    "Sports": [
        "Fitness",
        "Running",
        "Cricket",
        "Football",
        "Outdoor"
    ],
    "Grocery": [
        "Snacks",
        "Beverages",
        "Staples",
        "Dairy",
        "Packaged Food"
    ],
    "Books": [
        "Fiction",
        "Non-Fiction",
        "Technology",
        "Business",
        "Education"
    ],
    "Toys": [
        "Educational",
        "Action Figures",
        "Board Games",
        "Remote Control",
        "Kids Games"
    ]
}

brands = {
    "Electronics": [
        "Apple", "Samsung", "Sony", "OnePlus",
        "Dell", "HP", "Lenovo", "Boat"
    ],
    "Fashion": [
        "Nike", "Adidas", "Puma", "Levis",
        "Allen Solly", "Roadster"
    ],
    "Home & Kitchen": [
        "Philips", "Prestige", "Havells",
        "Ikea", "Bajaj", "Pigeon"
    ],
    "Beauty": [
        "Lakme", "Maybelline", "Loreal",
        "Nivea", "Dove", "Mamaearth"
    ],
    "Sports": [
        "Nike", "Adidas", "Puma",
        "Yonex", "Cosco", "SG"
    ],
    "Grocery": [
        "Nestle", "Britannia", "Tata",
        "Amul", "Parle", "Haldiram"
    ],
    "Books": [
        "Penguin", "HarperCollins",
        "Oxford", "McGraw Hill"
    ],
    "Toys": [
        "Lego", "Mattel", "Funskool",
        "Hasbro", "Hot Wheels"
    ]
}

# ------------------------------------------------------------
# Generate products
# ------------------------------------------------------------

product_data = []

product_start_date = datetime(2023, 1, 1)

for i in range(1, NUM_PRODUCTS + 1):

    product_id = f"P{i:05d}"

    category = random.choice(
        list(product_categories.keys())
    )

    subcategory = random.choice(
        product_categories[category]
    )

    brand = random.choice(
        brands[category]
    )

    product_name = f"{brand} {subcategory} Product {i}"

    # Generate realistic price ranges by category
    if category == "Electronics":
        price = builtins.round(random.uniform(1_000, 150_000), 2)

    elif category == "Fashion":
        price = builtins.round(random.uniform(300, 15_000), 2)

    elif category == "Home & Kitchen":
        price = builtins.round(random.uniform(200, 50_000), 2)

    elif category == "Beauty":
        price = builtins.round(random.uniform(100, 8_000), 2)

    elif category == "Sports":
        price = builtins.round(random.uniform(300, 20_000), 2)

    elif category == "Grocery":
        price = builtins.round(random.uniform(50, 5_000), 2)

    elif category == "Books":
        price = builtins.round(random.uniform(100, 3_000), 2)

    else:
        price = builtins.round(random.uniform(200, 10_000), 2)

    # Product cost between 55% and 85% of selling price
    cost = builtins.round(
        price * random.uniform(0.55, 0.85),
        2
    )

    supplier_id = f"SUP{random.randint(1, 500):04d}"

    product_rating = builtins.round(
        random.uniform(2.5, 5.0),
        1
    )

    created_date = (
        product_start_date +
        timedelta(
            days=random.randint(
                0,
                1_000
            )
        )
    ).date()

    updated_at = datetime(
        2026,
        random.randint(1, 7),
        random.randint(1, 28),
        random.randint(0, 23),
        random.randint(0, 59),
        random.randint(0, 59)
    )

    product_data.append((
        product_id,
        product_name,
        category,
        subcategory,
        brand,
        price,
        cost,
        supplier_id,
        product_rating,
        created_date,
        updated_at
    ))

print(f"Generated {len(product_data):,} products")

Generated 5,000 products


In [0]:
df_products = spark.createDataFrame(product_data, product_schema)

df_products.repartition(5).write.mode('overwrite').parquet(PRODUCT_PATH)

###Orders Data

In [0]:
# ============================================================
# ORDER DATA GENERATION
# ============================================================

NUM_ORDERS = 1_000_000

# Read the previously generated source data
customers_source = spark.read.parquet(CUSTOMER_PATH)
products_source = spark.read.parquet(PRODUCT_PATH)
NUM_CUSTOMERS = customers_source.count()
NUM_PRODUCTS = products_source.count()

customer_ids = customers_source.select("customer_id")
product_reference = products_source.select(
    "product_id",
    "price"
)

# ------------------------------------------------------------
# Generate base order IDs
# ------------------------------------------------------------

orders_df = (
    spark.range(1, NUM_ORDERS + 1)
    .withColumn(
        "order_id",
        F.format_string("O%08d", F.col("id"))
    )
    .drop("id")
)

# ------------------------------------------------------------
# Assign customers
# ------------------------------------------------------------

orders_df = (
    orders_df
    .withColumn(
        "customer_number",
        (
            F.floor(
                F.rand(RANDOM_SEED) * NUM_CUSTOMERS
            ) + 1
        ).cast("long")
    )
    .withColumn(
        "customer_id",
        F.format_string(
            "C%06d",
            F.col("customer_number")
        )
    )
    .drop("customer_number")
)

# ------------------------------------------------------------
# Assign products
# ------------------------------------------------------------

orders_df = (
    orders_df
    .withColumn(
        "product_number",
        (
            F.floor(
                F.rand(RANDOM_SEED + 1) * NUM_PRODUCTS
            ) + 1
        ).cast("long")
    )
    .withColumn(
        "product_id",
        F.format_string(
            "P%05d",
            F.col("product_number")
        )
    )
    .drop("product_number")
)

# ------------------------------------------------------------
# Join product price
# ------------------------------------------------------------

orders_df = (
    orders_df
    .join(
        F.broadcast(product_reference),
        on="product_id",
        how="left"
    )
)

# ------------------------------------------------------------
# Quantity
# ------------------------------------------------------------

orders_df = orders_df.withColumn(
    "quantity",
    (
        F.floor(
            F.rand(RANDOM_SEED + 2) * 5
        ) + 1
    ).cast("int")
)

# ------------------------------------------------------------
# Discount
# ------------------------------------------------------------

orders_df = orders_df.withColumn(
    "discount_percentage",
    F.round(
        F.rand(RANDOM_SEED + 3) * 30,
        2
    )
)

orders_df = orders_df.withColumn(
    "discount_amount",
    F.round(
        F.col("price")
        * F.col("quantity")
        * F.col("discount_percentage")
        / 100,
        2
    )
)

# ------------------------------------------------------------
# Order amount
# ------------------------------------------------------------

orders_df = orders_df.withColumn(
    "order_amount",
    F.round(
        (
            F.col("price") * F.col("quantity")
        ) - F.col("discount_amount"),
        2
    )
)

# ------------------------------------------------------------
# Payment method
# ------------------------------------------------------------

orders_df = orders_df.withColumn(
    "payment_method",
    F.expr("""
        CASE
            WHEN rand() < 0.40 THEN 'UPI'
            WHEN rand() < 0.65 THEN 'Credit Card'
            WHEN rand() < 0.80 THEN 'Debit Card'
            WHEN rand() < 0.90 THEN 'Net Banking'
            WHEN rand() < 0.97 THEN 'Wallet'
            ELSE 'COD'
        END
    """)
)

# ------------------------------------------------------------
# Order status
# ------------------------------------------------------------

orders_df = orders_df.withColumn(
    "order_status",
    F.expr("""
        CASE
            WHEN rand() < 0.70 THEN 'DELIVERED'
            WHEN rand() < 0.82 THEN 'SHIPPED'
            WHEN rand() < 0.90 THEN 'CONFIRMED'
            WHEN rand() < 0.95 THEN 'PLACED'
            WHEN rand() < 0.98 THEN 'CANCELLED'
            ELSE 'RETURNED'
        END
    """)
)

# ------------------------------------------------------------
# Order timestamp
# July 2025 - July 2026
# ------------------------------------------------------------

orders_df = orders_df.withColumn(
    "order_timestamp",
    F.expr("""
        timestampadd(
            SECOND,
            cast(rand() * 31536000 as int),
            timestamp('2025-07-01 00:00:00')
        )
    """)
)

# ------------------------------------------------------------
# Shipping city
# ------------------------------------------------------------

orders_df = orders_df.withColumn(
    "shipping_city",
    F.element_at(
        F.array(
            F.lit("Mumbai"),
            F.lit("Pune"),
            F.lit("Delhi"),
            F.lit("Bangalore"),
            F.lit("Hyderabad"),
            F.lit("Chennai"),
            F.lit("Kolkata"),
            F.lit("Ahmedabad"),
            F.lit("Jaipur"),
            F.lit("Surat"),
            F.lit("Nagpur"),
            F.lit("Nashik")
        ),
        (
            F.floor(
                F.rand(RANDOM_SEED + 4) * 12
            ) + 1
        ).cast("int")
    )
)

# ------------------------------------------------------------
# Updated timestamp
# ------------------------------------------------------------

orders_df = orders_df.withColumn(
    "updated_at",
    F.col("order_timestamp") +
    F.expr("INTERVAL 1 DAY")
)

# Select final columns
orders_df = orders_df.select(
    "order_id",
    "customer_id",
    "product_id",
    "quantity",
    "price",
    "discount_percentage",
    "discount_amount",
    "order_amount",
    "payment_method",
    "order_status",
    "order_timestamp",
    "shipping_city",
    "updated_at"
)

orders_df.printSchema()

root
 |-- order_id: string (nullable = false)
 |-- customer_id: string (nullable = false)
 |-- product_id: string (nullable = false)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- discount_percentage: double (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- order_amount: double (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- order_status: string (nullable = false)
 |-- order_timestamp: timestamp (nullable = true)
 |-- shipping_city: string (nullable = false)
 |-- updated_at: timestamp (nullable = true)



In [0]:
#display(orders_df.limit(20))
orders_df.repartition(20).write.mode("overwrite").parquet(ORDER_PATH)


###Payment Data

In [0]:
# ============================================================
# PAYMENT DATA GENERATION
# ============================================================

orders_for_payment = (
    spark.read
    .parquet(ORDER_PATH)
    .select(
        "order_id",
        "order_amount",
        "payment_method",
        "order_timestamp",
        "order_status"
    )
)

payments_df = (
    orders_for_payment
    .withColumn(
        "payment_id",
        F.concat(
            F.lit("PAY"),
            F.substring("order_id", 2, 8)
        )
    )
    .withColumn(
        "transaction_amount",
        F.col("order_amount")
    )
    .withColumn(
        "payment_status",
        F.when(
            F.col("order_status") == "CANCELLED",
            F.lit("FAILED")
        )
        .when(
            F.col("order_status") == "RETURNED",
            F.lit("REFUNDED")
        )
        .when(
            F.rand(RANDOM_SEED + 10) < 0.95,
            F.lit("SUCCESS")
        )
        .when(
            F.rand(RANDOM_SEED + 11) < 0.50,
            F.lit("PENDING")
        )
        .otherwise(
            F.lit("FAILED")
        )
    )
    .withColumn(
        "transaction_timestamp",
        F.col("order_timestamp") +
        F.expr(
            "INTERVAL 1 HOUR"
        )
    )
    .withColumn(
        "transaction_reference",
        F.concat(
            F.lit("TXN-"),
            F.upper(
                F.substring(
                    F.sha2(
                        F.col("order_id"),
                        256
                    ),
                    1,
                    12
                )
            )
        )
    )
    .select(
        "payment_id",
        "order_id",
        "payment_method",
        "payment_status",
        "transaction_amount",
        "transaction_timestamp",
        "transaction_reference"
    )
)

display(payments_df.limit(20))

payment_id,order_id,payment_method,payment_status,transaction_amount,transaction_timestamp,transaction_reference
PAY00000009,O00000009,UPI,SUCCESS,29978.6,2025-11-27T18:56:14.000Z,TXN-ED10CA65F6AD
PAY00000029,O00000029,Debit Card,SUCCESS,55347.94,2025-10-04T03:07:27.000Z,TXN-816CC79478C0
PAY00000049,O00000049,Credit Card,SUCCESS,102239.75,2025-12-13T15:45:25.000Z,TXN-9EB4118155F0
PAY00000069,O00000069,Debit Card,SUCCESS,102747.04,2025-08-12T21:55:42.000Z,TXN-A8DE085A5032
PAY00000089,O00000089,UPI,SUCCESS,79246.62,2026-01-29T15:44:55.000Z,TXN-E329F3F98B63
PAY00000109,O00000109,UPI,SUCCESS,6212.29,2025-12-10T07:44:38.000Z,TXN-821BB4A8DFC0
PAY00000129,O00000129,Debit Card,SUCCESS,1236.56,2025-12-24T16:13:15.000Z,TXN-842BADF9AA01
PAY00000149,O00000149,Net Banking,SUCCESS,619.03,2025-12-04T03:22:44.000Z,TXN-18CC1A4A3238
PAY00000169,O00000169,Debit Card,SUCCESS,13853.89,2026-03-23T14:48:19.000Z,TXN-0B5F445E794C
PAY00000189,O00000189,UPI,SUCCESS,34213.1,2026-01-09T10:23:08.000Z,TXN-30B9D830BCB7


In [0]:
payments_df.repartition(20).write.mode("overwrite").parquet(PAYMENT_PATH)

###Delivery Data

In [0]:
# ============================================================
# DELIVERY DATA GENERATION
# ============================================================

orders_for_delivery = (
    spark.read
    .parquet(ORDER_PATH)
    .filter(
        F.col("order_status").isin(
            "CONFIRMED",
            "SHIPPED",
            "DELIVERED",
            "RETURNED"
        )
    )
    .select(
        "order_id",
        "shipping_city",
        "order_timestamp",
        "order_status"
    )
)

deliveries_df = (
    orders_for_delivery
    .withColumn(
        "delivery_id",
        F.concat(
            F.lit("DEL"),
            F.substring("order_id", 2, 8)
        )
    )
    .withColumn(
        "delivery_partner",
        F.element_at(
            F.array(
                F.lit("QuickShip"),
                F.lit("FastTrack"),
                F.lit("BlueDart"),
                F.lit("Delhivery"),
                F.lit("EcomExpress")
            ),
            (
                F.floor(
                    F.rand(RANDOM_SEED + 20) * 5
                ) + 1
            ).cast("int")
        )
    )
    .withColumn(
        "warehouse",
        F.element_at(
            F.array(
                F.lit("WH-MUM"),
                F.lit("WH-PUN"),
                F.lit("WH-DEL"),
                F.lit("WH-BLR"),
                F.lit("WH-HYD")
            ),
            (
                F.floor(
                    F.rand(RANDOM_SEED + 21) * 5
                ) + 1
            ).cast("int")
        )
    )
)

In [0]:
# ============================================================
# DELIVERY DATES
# ============================================================

deliveries_df = (
    deliveries_df
    .withColumn(
        "shipped_date",
        F.col("order_timestamp") +
        F.expr(
            "INTERVAL 1 DAY"
        )
    )
    .withColumn(
        "estimated_delivery_date",
        F.col("order_timestamp") +
        F.expr(
            "INTERVAL 5 DAYS"
        )
    )
)

In [0]:
deliveries_df = (
    deliveries_df
    .withColumn(
        "actual_delivery_date",
        F.when(
            F.col("order_status") == "DELIVERED",
            F.col("order_timestamp") + F.expr("INTERVAL 1 DAY") * (2 + (F.rand(RANDOM_SEED + 25) * 6).cast("int"))
        )
    )
)

In [0]:
deliveries_df = (
    deliveries_df
    .withColumn(
        "delivery_status",
        F.when(
            F.col("order_status") == "DELIVERED",
            F.lit("DELIVERED")
        )
        .when(
            F.col("order_status") == "RETURNED",
            F.lit("RETURNED")
        )
        .when(
            F.col("order_status") == "SHIPPED",
            F.lit("IN_TRANSIT")
        )
        .otherwise(
            F.lit("PROCESSING")
        )
    )
)

In [0]:
deliveries_df = (
    deliveries_df
    .withColumn(
        "delivery_attempts",
        F.when(
            F.col("delivery_status") == "DELIVERED",
            F.floor(
                F.rand(RANDOM_SEED + 22) * 3
            ).cast("int") + 1
        )
        .otherwise(
            F.lit(0)
        )
    )
)

In [0]:
deliveries_df = deliveries_df.select(
    "delivery_id",
    "order_id",
    "delivery_partner",
    "warehouse",
    "shipping_city",
    "delivery_status",
    "order_timestamp",
    "shipped_date",
    "estimated_delivery_date",
    "actual_delivery_date",
    "delivery_attempts"
)

display(deliveries_df.limit(20))

delivery_id,order_id,delivery_partner,warehouse,shipping_city,delivery_status,order_timestamp,shipped_date,estimated_delivery_date,actual_delivery_date,delivery_attempts
DEL00000009,O00000009,QuickShip,WH-MUM,Nagpur,DELIVERED,2025-11-27T17:56:14.000Z,2025-11-28T17:56:14.000Z,2025-12-02T17:56:14.000Z,2025-12-02T17:56:14.000Z,2
DEL00000029,O00000029,Delhivery,WH-MUM,Nashik,DELIVERED,2025-10-04T02:07:27.000Z,2025-10-05T02:07:27.000Z,2025-10-09T02:07:27.000Z,2025-10-10T02:07:27.000Z,1
DEL00000049,O00000049,Delhivery,WH-PUN,Delhi,DELIVERED,2025-12-13T14:45:25.000Z,2025-12-14T14:45:25.000Z,2025-12-18T14:45:25.000Z,2025-12-19T14:45:25.000Z,2
DEL00000069,O00000069,BlueDart,WH-BLR,Mumbai,DELIVERED,2025-08-12T20:55:42.000Z,2025-08-13T20:55:42.000Z,2025-08-17T20:55:42.000Z,2025-08-15T20:55:42.000Z,2
DEL00000089,O00000089,FastTrack,WH-HYD,Nashik,DELIVERED,2026-01-29T14:44:55.000Z,2026-01-30T14:44:55.000Z,2026-02-03T14:44:55.000Z,2026-02-01T14:44:55.000Z,1
DEL00000109,O00000109,BlueDart,WH-DEL,Bangalore,DELIVERED,2025-12-10T06:44:38.000Z,2025-12-11T06:44:38.000Z,2025-12-15T06:44:38.000Z,2025-12-16T06:44:38.000Z,3
DEL00000129,O00000129,QuickShip,WH-MUM,Jaipur,DELIVERED,2025-12-24T15:13:15.000Z,2025-12-25T15:13:15.000Z,2025-12-29T15:13:15.000Z,2025-12-30T15:13:15.000Z,2
DEL00000149,O00000149,FastTrack,WH-PUN,Nagpur,DELIVERED,2025-12-04T02:22:44.000Z,2025-12-05T02:22:44.000Z,2025-12-09T02:22:44.000Z,2025-12-09T02:22:44.000Z,1
DEL00000169,O00000169,EcomExpress,WH-PUN,Surat,DELIVERED,2026-03-23T13:48:19.000Z,2026-03-24T13:48:19.000Z,2026-03-28T13:48:19.000Z,2026-03-29T13:48:19.000Z,3
DEL00000189,O00000189,FastTrack,WH-MUM,Ahmedabad,DELIVERED,2026-01-09T09:23:08.000Z,2026-01-10T09:23:08.000Z,2026-01-14T09:23:08.000Z,2026-01-16T09:23:08.000Z,3


In [0]:
deliveries_df.repartition(15).write.mode("overwrite").parquet(DELIVERY_PATH)

###Validation

In [0]:
customers = spark.read.parquet(CUSTOMER_PATH)
products = spark.read.parquet(PRODUCT_PATH)
orders = spark.read.parquet(ORDER_PATH)
payments = spark.read.parquet(PAYMENT_PATH)
deliveries = spark.read.parquet(DELIVERY_PATH)

print("Customers: ",customers.count())
print("Products: ",products.count())
print("Orders: ",orders.count())
print("Payments: ",payments.count())
print("Deliveries: ",deliveries.count())

Customers:  100000
Products:  5000
Orders:  1000000
Payments:  1000000
Deliveries:  994839


###Problematic Data

In [0]:
# ============================================================
# SOURCE-SYSTEM DATA ISSUES
# ============================================================

ISSUES_PATH = f"{SOURCE_VOLUME}/issues"

CUSTOMER_ISSUES_PATH = f"{ISSUES_PATH}/customers"
ORDER_ISSUES_PATH = f"{ISSUES_PATH}/orders"
PAYMENT_ISSUES_PATH = f"{ISSUES_PATH}/payments"
DELIVERY_ISSUES_PATH = f"{ISSUES_PATH}/deliveries"

print("Issues path:", ISSUES_PATH)

Issues path: /Volumes/quickcart/default/source_data/issues


In [0]:
# ============================================================
# ISSUE 1: DUPLICATE / UPDATED CUSTOMER RECORDS
# ============================================================

customers = spark.read.parquet(CUSTOMER_PATH)

# Select 5% of customers for updates
customer_updates = (
    customers
    .sample(
        withReplacement=False,
        fraction=0.05,
        seed=100
    )
    .withColumn(
        "city",
        F.element_at(
            F.array(
                F.lit("Mumbai"),
                F.lit("Pune"),
                F.lit("Bangalore"),
                F.lit("Delhi"),
                F.lit("Hyderabad")
            ),
            (
                F.floor(
                    F.rand(101) * 5
                ) + 1
            ).cast("int")
        )
    )
    .withColumn(
        "state",
        F.when(
            F.col("city") == "Mumbai",
            "Maharashtra"
        )
        .when(
            F.col("city") == "Pune",
            "Maharashtra"
        )
        .when(
            F.col("city") == "Bangalore",
            "Karnataka"
        )
        .when(
            F.col("city") == "Delhi",
            "Delhi"
        )
        .otherwise("Telangana")
    )
    .withColumn(
        "updated_at",
        F.current_timestamp()
    )
)

print(
    "Customer update records:",
    customer_updates.count()
)

display(customer_updates.limit(10))

Customer update records: 4977


customer_id,customer_name,email,phone,gender,date_of_birth,city,state,pincode,registration_date,customer_segment,updated_at
C000012,Suresh Verma,suresh.verma12@quickcart.com,9919706735,Male,2000-05-27,Pune,Maharashtra,302001,2024-11-05,Regular,2026-08-10T11:24:32.578Z
C000102,Raj Mishra,raj.mishra102@quickcart.com,9735313433,Male,1981-03-26,Delhi,Delhi,380001,2022-01-06,Regular,2026-08-10T11:24:32.578Z
C000472,Anjali Shah,anjali.shah472@quickcart.com,6626285594,Female,1962-04-07,Delhi,Delhi,302001,2024-02-05,Regular,2026-08-10T11:24:32.578Z
C000552,Riya Sharma,riya.sharma552@quickcart.com,8213553149,Female,1990-07-13,Mumbai,Maharashtra,395001,2026-05-19,Premium,2026-08-10T11:24:32.578Z
C000802,Rohan Pawar,rohan.pawar802@quickcart.com,9056339391,Female,1985-12-19,Pune,Maharashtra,411001,2024-11-27,Premium,2026-08-10T11:24:32.578Z
C000862,Arjun Pawar,arjun.pawar862@quickcart.com,9524490597,Female,1972-08-23,Hyderabad,Telangana,411001,2025-12-06,Premium,2026-08-10T11:24:32.578Z
C000902,Suresh Verma,suresh.verma902@quickcart.com,6457261738,Male,1973-10-01,Hyderabad,Telangana,110001,2023-01-20,Regular,2026-08-10T11:24:32.578Z
C000912,Amit Shah,amit.shah912@quickcart.com,8765386801,Male,1975-12-27,Mumbai,Maharashtra,700001,2022-01-17,Regular,2026-08-10T11:24:32.578Z
C001172,Pooja Reddy,pooja.reddy1172@quickcart.com,6092271071,Female,1996-05-01,Mumbai,Maharashtra,380001,2024-03-09,Regular,2026-08-10T11:24:32.578Z
C001212,Suresh Mishra,suresh.mishra1212@quickcart.com,8146781020,Male,1981-06-20,Pune,Maharashtra,700001,2026-02-24,Premium,2026-08-10T11:24:32.578Z


In [0]:
customer_updates.repartition(2).write.mode("overwrite").parquet(f"{CUSTOMER_ISSUES_PATH}/updates")

In [0]:
# ============================================================
# ISSUE 2: DUPLICATE / UPDATED ORDERS
# ============================================================

orders = spark.read.parquet(ORDER_PATH)

order_updates = (
    orders
    .sample(
        withReplacement=False,
        fraction=0.02,
        seed=200
    )
    .withColumn(
        "order_status",
        F.when(
            F.col("order_status") == "PLACED",
            "CONFIRMED"
        )
        .when(
            F.col("order_status") == "CONFIRMED",
            "SHIPPED"
        )
        .when(
            F.col("order_status") == "SHIPPED",
            "DELIVERED"
        )
        .otherwise(
            F.col("order_status")
        )
    )
    .withColumn(
        "updated_at",
        F.col("updated_at") +
        F.expr("INTERVAL 2 HOURS")
    )
)

print(
    "Order update records:",
    order_updates.count()
)

display(order_updates.limit(10))

order_updates.repartition(5).write.mode("overwrite").parquet(f"{ORDER_ISSUES_PATH}/updates")

Order update records: 20169


order_id,customer_id,product_id,quantity,price,discount_percentage,discount_amount,order_amount,payment_method,order_status,order_timestamp,shipping_city,updated_at
O00000812,C020400,P01928,5,751.71,7.61,286.03,3472.52,UPI,DELIVERED,2026-05-12T05:13:02.000Z,Surat,2026-05-13T07:13:02.000Z
O00001352,C033657,P04453,3,6757.23,29.63,6006.5,14265.19,UPI,DELIVERED,2026-04-01T13:04:11.000Z,Ahmedabad,2026-04-02T15:04:11.000Z
O00001492,C031103,P00936,3,4585.36,22.12,3042.84,10713.24,UPI,DELIVERED,2025-09-11T16:43:35.000Z,Hyderabad,2025-09-12T18:43:35.000Z
O00006112,C002281,P02862,4,3127.55,23.32,2917.38,9592.82,Debit Card,DELIVERED,2026-03-30T18:17:48.000Z,Delhi,2026-03-31T20:17:48.000Z
O00006552,C050979,P00194,5,1986.56,19.83,1969.67,7963.13,Debit Card,DELIVERED,2026-05-18T04:38:56.000Z,Surat,2026-05-19T06:38:56.000Z
O00006932,C022012,P01483,4,17824.85,0.72,513.36,70786.04,Credit Card,DELIVERED,2026-03-24T22:08:36.000Z,Nashik,2026-03-26T00:08:36.000Z
O00007352,C056263,P01579,5,97125.2,12.03,58420.81,427205.19,UPI,DELIVERED,2025-07-23T15:59:25.000Z,Kolkata,2025-07-24T17:59:25.000Z
O00008552,C009671,P04062,4,14579.71,4.72,2752.65,55566.19,Credit Card,DELIVERED,2025-11-25T13:26:58.000Z,Surat,2025-11-26T15:26:58.000Z
O00009252,C053790,P01502,4,1404.14,19.71,1107.02,4509.54,Debit Card,DELIVERED,2025-07-08T23:01:40.000Z,Jaipur,2025-07-10T01:01:40.000Z
O00009652,C060789,P01938,1,10946.04,29.27,3203.91,7742.13,Credit Card,DELIVERED,2026-03-21T11:47:04.000Z,Delhi,2026-03-22T13:47:04.000Z


In [0]:
# ============================================================
# ISSUE 3: NULL VALUES
# ============================================================

customer_nulls = (
    customers
    .sample(
        withReplacement=False,
        fraction=0.03,
        seed=300
    )
    .withColumn(
        "phone",
        F.lit(None).cast("string")
    )
    .withColumn(
        "email",
        F.lit(None).cast("string")
    )
)

print(
    "Customers with NULL attributes:",
    customer_nulls.count()
)

display(customer_nulls.limit(10))

customer_nulls.repartition(2).write.mode("overwrite").parquet(f"{CUSTOMER_ISSUES_PATH}/null_records")

Customers with NULL attributes: 3005


customer_id,customer_name,email,phone,gender,date_of_birth,city,state,pincode,registration_date,customer_segment,updated_at
C000012,Suresh Verma,null,null,Male,2000-05-27,Jaipur,Rajasthan,302001,2024-11-05,Regular,2026-02-12T05:34:49.000Z
C000162,Aarti Pawar,null,null,Male,1970-11-20,Surat,Gujarat,395001,2025-05-24,Regular,2026-05-02T00:05:02.000Z
C000262,Sneha Desai,null,null,Female,1968-04-09,Jaipur,Rajasthan,302001,2022-06-22,Regular,2026-05-20T23:38:18.000Z
C000752,Akash Joshi,null,null,Male,1981-04-10,Chennai,Tamil Nadu,600001,2023-05-07,Regular,2026-03-22T05:33:51.000Z
C000812,Raj Sharma,null,null,Female,1961-11-12,Mumbai,Maharashtra,400001,2026-07-12,Premium,2026-06-15T07:38:31.000Z
C001002,Anjali Joshi,null,null,Female,1964-10-03,Nagpur,Maharashtra,440001,2026-05-22,Regular,2026-01-04T15:14:03.000Z
C001012,Neha Shah,null,null,Male,1978-03-20,Hyderabad,Telangana,500001,2024-09-07,Regular,2026-06-03T02:16:34.000Z
C001122,Priya Patil,null,null,Male,1961-12-11,Nashik,Maharashtra,422001,2022-05-11,Regular,2026-06-23T19:52:29.000Z
C001502,Kavya Joshi,null,null,Male,1999-04-17,Delhi,Delhi,110001,2023-08-18,Premium,2026-01-04T07:19:57.000Z
C001892,Sneha Gupta,null,null,Female,1988-12-13,Nagpur,Maharashtra,440001,2025-04-28,Premium,2026-05-03T09:14:13.000Z


In [0]:
# ============================================================
# ISSUE 4A: INVALID CUSTOMER REFERENCES
# ============================================================

invalid_customer_orders = (
    orders
    .sample(
        withReplacement=False,
        fraction=0.005,
        seed=400
    )
    .withColumn(
        "customer_id",
        F.concat(
            F.lit("INVALID_C"),
            F.substring(
                F.col("order_id"),
                2,
                8
            )
        )
    )
)

print(
    "Orders with invalid customers:",
    invalid_customer_orders.count()
)

display(
    invalid_customer_orders.limit(10)
)

invalid_customer_orders.repartition(2).write.mode("overwrite").parquet(f"{ORDER_ISSUES_PATH}/invalid_customers")

Orders with invalid customers: 5032


order_id,customer_id,product_id,quantity,price,discount_percentage,discount_amount,order_amount,payment_method,order_status,order_timestamp,shipping_city,updated_at
O00002952,INVALID_C00002952,P01273,3,11302.01,3.59,1217.23,32688.8,Debit Card,DELIVERED,2026-02-22T17:19:38.000Z,Nashik,2026-02-23T17:19:38.000Z
O00007732,INVALID_C00007732,P02967,5,6561.39,12.18,3995.89,28811.06,UPI,DELIVERED,2025-09-19T20:00:52.000Z,Bangalore,2025-09-20T20:00:52.000Z
O00008532,INVALID_C00008532,P03197,2,446.48,5.45,48.67,844.29,Credit Card,DELIVERED,2026-06-04T03:49:23.000Z,Surat,2026-06-05T03:49:23.000Z
O00008572,INVALID_C00008572,P00221,4,3832.91,0.44,67.46,15264.18,Credit Card,SHIPPED,2026-02-18T09:12:32.000Z,Kolkata,2026-02-19T09:12:32.000Z
O00011912,INVALID_C00011912,P00586,2,5445.98,18.58,2023.73,8868.23,Credit Card,DELIVERED,2025-08-26T10:37:36.000Z,Hyderabad,2025-08-27T10:37:36.000Z
O00016092,INVALID_C00016092,P03836,3,12841.49,20.15,7762.68,30761.79,Credit Card,DELIVERED,2026-05-27T10:08:49.000Z,Ahmedabad,2026-05-28T10:08:49.000Z
O00025052,INVALID_C00025052,P01815,5,8051.3,23.36,9403.92,30852.58,Debit Card,DELIVERED,2025-12-03T01:44:22.000Z,Nagpur,2025-12-04T01:44:22.000Z
O00026152,INVALID_C00026152,P01087,3,1120.76,16.92,568.9,2793.38,UPI,CONFIRMED,2026-04-23T03:18:16.000Z,Chennai,2026-04-24T03:18:16.000Z
O00032552,INVALID_C00032552,P04494,5,1174.78,12.76,749.51,5124.39,Credit Card,DELIVERED,2025-08-21T20:58:33.000Z,Surat,2025-08-22T20:58:33.000Z
O00041012,INVALID_C00041012,P00474,2,13983.48,25.35,7089.62,20877.34,UPI,DELIVERED,2026-02-15T08:29:27.000Z,Nashik,2026-02-16T08:29:27.000Z


In [0]:
# ============================================================
# ISSUE 4B: INVALID PRODUCT REFERENCES
# ============================================================

invalid_product_orders = (
    orders
    .sample(
        withReplacement=False,
        fraction=0.005,
        seed=500
    )
    .withColumn(
        "product_id",
        F.concat(
            F.lit("INVALID_P"),
            F.substring(
                F.col("order_id"),
                2,
                8
            )
        )
    )
)

print(
    "Orders with invalid products:",
    invalid_product_orders.count()
)

display(
    invalid_product_orders.limit(10)
)

invalid_product_orders.repartition(2).write.mode("overwrite").parquet(f"{ORDER_ISSUES_PATH}/invalid_products")

Orders with invalid products: 4999


order_id,customer_id,product_id,quantity,price,discount_percentage,discount_amount,order_amount,payment_method,order_status,order_timestamp,shipping_city,updated_at
O00008352,C004061,INVALID_P00008352,2,3216.55,21.55,1386.33,5046.77,Credit Card,SHIPPED,2025-08-23T10:55:17.000Z,Mumbai,2025-08-24T10:55:17.000Z
O00010952,C044092,INVALID_P00010952,1,399.55,22.86,91.34,308.21,UPI,DELIVERED,2025-11-21T21:52:55.000Z,Nashik,2025-11-22T21:52:55.000Z
O00011072,C086168,INVALID_P00011072,3,40317.0,9.41,11381.49,109569.51,Credit Card,DELIVERED,2025-10-15T18:23:30.000Z,Jaipur,2025-10-16T18:23:30.000Z
O00017992,C015436,INVALID_P00017992,1,6207.78,20.76,1288.74,4919.04,Credit Card,SHIPPED,2025-11-24T00:09:50.000Z,Nagpur,2025-11-25T00:09:50.000Z
O00018032,C066943,INVALID_P00018032,1,1261.46,29.5,372.13,889.33,Credit Card,DELIVERED,2025-09-25T05:13:36.000Z,Hyderabad,2025-09-26T05:13:36.000Z
O00025252,C025417,INVALID_P00025252,5,1702.13,10.89,926.81,7583.84,Credit Card,DELIVERED,2025-07-03T00:25:01.000Z,Nashik,2025-07-04T00:25:01.000Z
O00026712,C021752,INVALID_P00026712,4,313.3,8.27,103.64,1149.56,Credit Card,DELIVERED,2025-10-09T09:17:58.000Z,Surat,2025-10-10T09:17:58.000Z
O00026892,C069788,INVALID_P00026892,3,5966.98,28.09,5028.37,12872.57,Credit Card,DELIVERED,2025-10-07T00:09:57.000Z,Jaipur,2025-10-08T00:09:57.000Z
O00031252,C055271,INVALID_P00031252,3,1634.43,15.74,771.78,4131.51,Debit Card,DELIVERED,2026-02-12T04:29:38.000Z,Jaipur,2026-02-13T04:29:38.000Z
O00032732,C043630,INVALID_P00032732,5,3562.2,27.33,4867.75,12943.25,UPI,DELIVERED,2026-06-02T09:10:16.000Z,Mumbai,2026-06-03T09:10:16.000Z


In [0]:
# ============================================================
# ISSUE 5: INVALID BUSINESS VALUES
# ============================================================

invalid_business_orders = (
    orders
    .sample(
        withReplacement=False,
        fraction=0.005,
        seed=600
    )
    .withColumn(
        "quantity",
        F.when(
            F.rand(601) < 0.33,
            F.lit(-1)
        )
        .when(
            F.rand(602) < 0.66,
            F.lit(0)
        )
        .otherwise(
            F.lit(1)
        )
    )
    .withColumn(
        "discount_percentage",
        F.when(
            F.rand(603) < 0.5,
            F.lit(150.0)
        )
        .otherwise(
            F.lit(-20.0)
        )
    )
    .withColumn(
        "order_amount",
        F.when(
            F.rand(604) < 0.5,
            F.lit(-500.0)
        )
        .otherwise(
            F.lit(0.0)
        )
    )
)

display(
    invalid_business_orders.limit(10)
)

invalid_business_orders.repartition(2).write.mode("overwrite").parquet(f"{ORDER_ISSUES_PATH}/invalid_business_values")

order_id,customer_id,product_id,quantity,price,discount_percentage,discount_amount,order_amount,payment_method,order_status,order_timestamp,shipping_city,updated_at
O00000312,C040677,P01300,0,8576.05,-20.0,9296.44,-500.0,Debit Card,DELIVERED,2025-08-17T06:16:51.000Z,Jaipur,2025-08-18T06:16:51.000Z
O00011932,C068522,P00145,0,5673.99,150.0,119.72,0.0,UPI,DELIVERED,2025-09-04T13:32:53.000Z,Ahmedabad,2025-09-05T13:32:53.000Z
O00016432,C058922,P01495,-1,26199.92,150.0,23936.25,0.0,Debit Card,DELIVERED,2025-12-28T04:58:52.000Z,Mumbai,2025-12-29T04:58:52.000Z
O00019792,C057598,P03180,-1,1280.37,150.0,55.31,0.0,Credit Card,DELIVERED,2026-02-10T04:18:01.000Z,Chennai,2026-02-11T04:18:01.000Z
O00024732,C053068,P02209,0,5274.15,-20.0,3692.96,0.0,Debit Card,DELIVERED,2026-01-30T09:09:48.000Z,Hyderabad,2026-01-31T09:09:48.000Z
O00026792,C003724,P03624,1,3050.66,150.0,1524.72,0.0,Net Banking,DELIVERED,2025-09-14T02:37:11.000Z,Jaipur,2025-09-15T02:37:11.000Z
O00032192,C019629,P04509,-1,145990.9,150.0,64513.38,-500.0,UPI,SHIPPED,2025-10-13T19:53:20.000Z,Jaipur,2025-10-14T19:53:20.000Z
O00032492,C030358,P03082,1,4019.74,150.0,693.41,-500.0,Debit Card,SHIPPED,2025-08-25T22:30:40.000Z,Bangalore,2025-08-26T22:30:40.000Z
O00037672,C057467,P03851,0,1245.83,150.0,405.64,-500.0,UPI,DELIVERED,2026-04-20T14:52:32.000Z,Delhi,2026-04-21T14:52:32.000Z
O00038092,C062087,P04642,-1,67049.24,-20.0,10325.58,0.0,Credit Card,DELIVERED,2026-02-14T19:31:41.000Z,Pune,2026-02-15T19:31:41.000Z


In [0]:
# ============================================================
# ISSUE 6: LATE ARRIVING ORDERS
# ============================================================

late_orders = (
    orders
    .sample(
        withReplacement=False,
        fraction=0.005,
        seed=700
    )
    .withColumn(
        "order_timestamp",
        F.to_timestamp(
            F.lit("2026-07-10 10:00:00")
        )
    )
    .withColumn(
        "updated_at",
        F.to_timestamp(
            F.lit("2026-07-11 15:00:00")
        )
    )
)

display(
    late_orders.limit(10)
)

late_orders.repartition(2).write.mode("overwrite").parquet(f"{ORDER_ISSUES_PATH}/late_arrivals")

order_id,customer_id,product_id,quantity,price,discount_percentage,discount_amount,order_amount,payment_method,order_status,order_timestamp,shipping_city,updated_at
O00009072,C058951,P01837,2,2468.53,12.16,600.35,4336.71,Credit Card,PLACED,2026-07-10T10:00:00.000Z,Pune,2026-07-11T15:00:00.000Z
O00009852,C092047,P04167,3,33266.16,3.62,3612.7,96185.78,UPI,DELIVERED,2026-07-10T10:00:00.000Z,Surat,2026-07-11T15:00:00.000Z
O00010052,C034772,P02050,5,34338.45,13.56,23281.47,148410.78,Credit Card,SHIPPED,2026-07-10T10:00:00.000Z,Jaipur,2026-07-11T15:00:00.000Z
O00011632,C081842,P02265,5,1346.09,29.2,1965.29,4765.16,UPI,DELIVERED,2026-07-10T10:00:00.000Z,Delhi,2026-07-11T15:00:00.000Z
O00012132,C091073,P02605,2,105690.11,4.98,10526.73,200853.49,Credit Card,DELIVERED,2026-07-10T10:00:00.000Z,Surat,2026-07-11T15:00:00.000Z
O00018472,C048313,P00629,4,830.6,9.99,331.91,2990.49,Credit Card,SHIPPED,2026-07-10T10:00:00.000Z,Bangalore,2026-07-11T15:00:00.000Z
O00019452,C042751,P01937,3,2685.61,0.78,62.84,7993.99,Credit Card,DELIVERED,2026-07-10T10:00:00.000Z,Jaipur,2026-07-11T15:00:00.000Z
O00019472,C028814,P03785,2,7959.63,14.93,2376.75,13542.51,UPI,DELIVERED,2026-07-10T10:00:00.000Z,Jaipur,2026-07-11T15:00:00.000Z
O00020472,C047886,P03198,5,7069.96,14.43,5100.98,30248.82,Credit Card,DELIVERED,2026-07-10T10:00:00.000Z,Hyderabad,2026-07-11T15:00:00.000Z
O00023072,C039286,P03736,5,11933.7,4.4,2625.41,57043.09,Credit Card,SHIPPED,2026-07-10T10:00:00.000Z,Ahmedabad,2026-07-11T15:00:00.000Z


In [0]:
# ============================================================
# ISSUE 7: NULL TRANSACTION ATTRIBUTES
# ============================================================

order_nulls = (
    orders
    .sample(
        withReplacement=False,
        fraction=0.005,
        seed=800
    )
    .withColumn(
        "customer_id",
        F.lit(None).cast("string")
    )
    .withColumn(
        "payment_method",
        F.lit(None).cast("string")
    )
    .withColumn(
        "shipping_city",
        F.lit(None).cast("string")
    )
)

display(
    order_nulls.limit(10)
)

order_nulls.repartition(2).write.mode("overwrite").parquet(f"{ORDER_ISSUES_PATH}/null_records")

order_id,customer_id,product_id,quantity,price,discount_percentage,discount_amount,order_amount,payment_method,order_status,order_timestamp,shipping_city,updated_at
O00000152,null,P01805,3,6801.4,1.17,238.73,20165.47,null,DELIVERED,2025-12-22T12:10:16.000Z,null,2025-12-23T12:10:16.000Z
O00016452,null,P04984,4,7100.91,29.66,8424.52,19979.12,null,DELIVERED,2026-05-13T17:53:00.000Z,null,2026-05-14T17:53:00.000Z
O00018632,null,P02805,2,1814.95,4.16,151.0,3478.9,null,DELIVERED,2026-06-11T18:50:57.000Z,null,2026-06-12T18:50:57.000Z
O00020492,null,P04966,4,31898.39,7.9,10079.89,117513.67,null,DELIVERED,2025-08-10T12:36:58.000Z,null,2025-08-11T12:36:58.000Z
O00033272,null,P03164,4,372.26,3.96,58.97,1430.07,null,CONFIRMED,2026-05-28T19:36:59.000Z,null,2026-05-29T19:36:59.000Z
O00036052,null,P02256,2,539.46,19.58,211.25,867.67,null,DELIVERED,2026-05-17T21:16:15.000Z,null,2026-05-18T21:16:15.000Z
O00037892,null,P03100,4,54727.28,16.65,36448.37,182460.75,null,DELIVERED,2026-01-26T06:06:12.000Z,null,2026-01-27T06:06:12.000Z
O00039052,null,P03649,4,382.74,17.44,267.0,1263.96,null,DELIVERED,2025-10-02T20:37:21.000Z,null,2025-10-03T20:37:21.000Z
O00039232,null,P04789,5,19876.76,2.61,2593.92,96789.88,null,DELIVERED,2025-11-04T23:39:13.000Z,null,2025-11-05T23:39:13.000Z
O00040452,null,P02194,1,3986.37,26.15,1042.44,2943.93,null,DELIVERED,2025-11-16T19:22:22.000Z,null,2025-11-17T19:22:22.000Z


In [0]:
# ============================================================
# SOURCE ISSUE SUMMARY
# ============================================================

issue_summary = [
    ("Duplicate/Updated Customers", customer_updates.count()),
    ("Customer NULL Records", customer_nulls.count()),
    ("Duplicate/Updated Orders", order_updates.count()),
    ("Invalid Customer References", invalid_customer_orders.count()),
    ("Invalid Product References", invalid_product_orders.count()),
    ("Invalid Business Values", invalid_business_orders.count()),
    ("Late Arriving Orders", late_orders.count()),
    ("Order NULL Records", order_nulls.count())
]

issue_summary_df = spark.createDataFrame(
    issue_summary,
    ["issue_type", "record_count"]
)

display(issue_summary_df)

issue_type,record_count
Duplicate/Updated Customers,4977
Customer NULL Records,3005
Duplicate/Updated Orders,20169
Invalid Customer References,5032
Invalid Product References,4999
Invalid Business Values,5052
Late Arriving Orders,5065
Order NULL Records,4959
